## SAE (with SAEManagerEdit)

This notebook demonstrates the SAE workflow using the new `SAEManagerEdit` class.

**Key differences from walkthrough.ipynb:**
- Uses `SAEManagerEdit` instead of `SAEManager` (hooks → .edit())
- SAE encoder/decoder are proper Envoys (traceable)
- Can directly intervene on SAE features without hooks
- Works on MPS (Apple Silicon) and CUDA

In [ ]:
import os

os.chdir("..")  # should be run from repo root
from dictionary_learning.trainers.top_k import AutoEncoderTopK

#### update sae state_dict to match dictionary_learning implementation

In [ ]:
import torch

# Device selection
if torch.cuda.is_available():
    DEVICE = "cuda:0"
elif torch.backends.mps.is_available():
    DEVICE = "mps"
else:
    DEVICE = "cpu"
print(f"Using device: {DEVICE}")

# Convert checkpoint if needed (one-time operation)
checkpoint_base = "./sdxl-unbox/checkpoints_hf/unet.down_blocks.2.attentions.1_k10_hidden5120_auxk256_bs4096_lr0.0001"
raw_path = f"{checkpoint_base}/state_dict.pth"
final_path = f"{checkpoint_base}/final/state_dict.pth"

os.makedirs(f"{checkpoint_base}/final", exist_ok=True)

if not os.path.exists(final_path):
    print("Converting checkpoint...")
    sd = torch.load(raw_path, map_location=DEVICE, weights_only=False)
    sd = sd["state_dict"]
    sd = {
        k.replace("pre_bias", "b_dec"): v
        for k, v in sd.items()
        if k in ["encoder.weight", "pre_bias", "decoder.weight"]
    }
    sd.update(
        {
            "encoder.bias": torch.zeros((5120,)),
            "k": torch.tensor(10),
            "threshold": torch.tensor(-1.0),
        }
    )
    torch.save(sd, final_path)
    print(f"Saved converted checkpoint to: {final_path}")
else:
    print(f"Using existing converted checkpoint: {final_path}")

In [ ]:
dtype = torch.float16
sae1 = AutoEncoderTopK.from_pretrained(
    final_path,
    k=10,
    device=DEVICE,
).to(dtype)
print(f"Loaded SAE: dict_size={sae1.dict_size}, activation_dim={sae1.activation_dim}")

In [ ]:
# NEW: Use SAEManagerEdit instead of SAEManager
from t2Interp.sae_edit import SAEManagerEdit
from t2Interp.T2I import T2IModel

model = T2IModel("stabilityai/sdxl-turbo", device=DEVICE, dtype="float16")
print(f"Model loaded on: {model.device}")

### add SAEs to the model

The new `SAEManagerEdit` uses the same interface as the old `SAEManager`.

In [ ]:
# NEW: Use SAEManagerEdit
sae_manager = SAEManagerEdit(model=model)
kwargs = {"diff": True}  # SAEs trained on residuals (output - input)
sae1.k = torch.Tensor([5120]).to(DEVICE, dtype=torch.int32)

# Add SAE to the model (same interface as old SAEManager)
sae_blocks = sae_manager.add_saes_to_model(
    sae_list=[(model.unet_2.down_attn_blocks[7].cross_attn_out, sae1, "cross_attn_out")], **kwargs
)
sae_block = sae_blocks[0]  # Get the SAE block
print(f"SAEs added: {sae_manager.sae_names}")

### record activating features on a prompt

**Key difference:** With `SAEManagerEdit`, we can trace SAE encoder output directly.

In [ ]:
# Apply edits to wire SAE into the model
edited_model = sae_manager.apply_edits()

# Get the SAE block for accessing encoder/decoder

prompt = "A cinematic shot of a professor sloth wearing a tuxedo at a BBQ party."

# NEW: Directly trace SAE encoder output (no hooks needed!)
with edited_model.generate(
    prompt,
    num_inference_steps=1,
    guidance_scale=0.0,
    seed=42,
    validate=False,
    scan=False,
):
    # Access SAE encoder output directly via the SAE block
    sparse_maps = sae_block.encoder_out.value.save()
    output = edited_model.output.save()

print(f"Encoder output shape: {sparse_maps.shape}")
print(f"Generated image")

In [ ]:
# Find top activating features
top_features = sparse_maps.mean(axis=(0, 1)).topk(10).indices.cpu().tolist()
print(f"Top 10 features: {top_features}")

### visualize feature heatmap for sample features

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.colors import ListedColormap
from PIL import Image


def plot_image_heatmap(image, sparse_maps, feature):
    """Plot image with SAE feature heatmap overlay."""
    heatmap = sparse_maps[:, :, feature].cpu().numpy()
    heatmap = np.kron(heatmap, np.ones((32, 32)))
    image = image.convert("RGBA")

    jet = plt.cm.jet
    cmap = jet(np.arange(jet.N))
    cmap[:1, -1] = 0
    cmap[1:, -1] = 0.6
    cmap = ListedColormap(cmap)
    heatmap = (heatmap - np.min(heatmap)) / (np.max(heatmap) - np.min(heatmap) + 1e-8)

    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    axes[0].imshow(image)
    axes[0].set_title("Original Image")
    axes[0].axis("off")

    axes[1].imshow(heatmap, cmap="jet")
    axes[1].set_title(f"Feature {feature} Heatmap")
    axes[1].axis("off")

    axes[2].imshow(image)
    axes[2].imshow(heatmap, cmap=cmap, alpha=0.6)
    axes[2].set_title(f"Feature {feature} Overlay")
    axes[2].axis("off")

    plt.tight_layout()
    plt.show()


# Plot heatmaps for top features
for feature in top_features[:3]:
    plot_image_heatmap(output.images[0], sparse_maps[0], feature)

### ablate selected features on the prompt

**Key difference:** With `SAEManagerEdit`, we can directly modify SAE encoder output in the trace context.

In [ ]:
def generate_with_feature_intervention(features_to_ablate, scale_factor=0.0):
    """
    Generate image with specific SAE features scaled.

    Args:
        features_to_ablate: List of feature indices to scale
        scale_factor: 0.0 = ablate, 2.0 = amplify
    """
    with edited_model.generate(
        prompt,
        num_inference_steps=1,
        guidance_scale=0.0,
        seed=42,
        validate=False,
        scan=False,
    ):
        # Get encoder output
        enc_out = sae_block.encoder_out.value

        # Scale specific features
        for idx in features_to_ablate:
            enc_out[:, :, idx] = enc_out[:, :, idx] * scale_factor

        # Apply the modification
        sae_block.encoder_out.value = enc_out

        result = edited_model.output.save()

    return result.images[0]

In [ ]:
# Generate baseline
baseline_image = output.images[0]

# Ablate top feature
top_feature = top_features[0]
ablated_image = generate_with_feature_intervention([top_feature], scale_factor=0.0)

# Amplify top feature
amplified_image = generate_with_feature_intervention([top_feature], scale_factor=2.0)

print(f"Feature {top_feature}: baseline / ablated / amplified")

In [ ]:
# Compare results
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

axes[0].imshow(baseline_image)
axes[0].set_title("Baseline")
axes[0].axis("off")

axes[1].imshow(ablated_image)
axes[1].set_title(f"Feature {top_feature} ablated (0x)")
axes[1].axis("off")

axes[2].imshow(amplified_image)
axes[2].set_title(f"Feature {top_feature} amplified (2x)")
axes[2].axis("off")

plt.tight_layout()
plt.show()

### feature sparsity plot

In [ ]:
# Analyze feature activation distribution
activations = sparse_maps.float().cpu().numpy()
feature_activations = np.abs(activations).sum(axis=(0, 1))  # Sum over batch and sequence

# Sort by activation strength
sorted_indices = np.argsort(feature_activations)[::-1]
sorted_activations = feature_activations[sorted_indices]

# Plot activation distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# All features
axes[0].bar(range(len(sorted_activations)), sorted_activations)
axes[0].set_xlabel("Feature Rank")
axes[0].set_ylabel("Total Activation")
axes[0].set_title("Feature Activation Distribution (sorted)")

# Top 50 features
axes[1].bar(range(50), sorted_activations[:50])
axes[1].set_xlabel("Feature Rank")
axes[1].set_ylabel("Total Activation")
axes[1].set_title("Top 50 Features")

plt.tight_layout()
plt.show()

# Print sparsity statistics
nonzero = (feature_activations > 0).sum()
print(
    f"Active features: {nonzero}/{len(feature_activations)} ({100*nonzero/len(feature_activations):.1f}%)"
)

### using intervention classes

You can also use the intervention classes with the SAE accessors.

In [ ]:
from t2Interp.intervention import FeatureIntervention, run_intervention

# Create intervention on SAE encoder output
intervention = FeatureIntervention(
    model=edited_model,
    accessors=[sae_block.encoder_out],
    features=top_features[:3],  # Intervene on top 3 features
    scale=0.0,  # Ablate
)

print(f"Created intervention on features: {top_features[:3]}")

### comparison with multiple SAEs

You can add multiple SAEs to different layers.

In [ ]:
# Example: Adding multiple SAEs (uncomment to use)
#
# # Convert additional checkpoints first
# for checkpoint_name in [
#     "unet.mid_block.attentions.0_k10_hidden5120_auxk256_bs4096_lr0.0001",
#     "unet.up_blocks.0.attentions.0_k10_hidden5120_auxk256_bs4096_lr0.0001",
#     "unet.up_blocks.0.attentions.1_k10_hidden5120_auxk256_bs4096_lr0.0001",
# ]:
#     checkpoint_base = f"./sdxl-unbox/checkpoints_hf/{checkpoint_name}"
#     # ... convert checkpoint ...
#
# # Load SAEs
# sae2 = AutoEncoderTopK.from_pretrained(...)
# sae3 = AutoEncoderTopK.from_pretrained(...)
# sae4 = AutoEncoderTopK.from_pretrained(...)
#
# # Add multiple SAEs
# sae_blocks = sae_manager.add_saes_to_model(
#     sae_list=[
#         (model.unet_2.down_attn_blocks[7].cross_attn_out, sae1, "down_cross"),
#         (model.unet_2.mid_attn_block.self_attn_out, sae2, "mid_self"),
#         (model.unet_2.up_attn_blocks[0].self_attn_out, sae3, "up0_self"),
#         (model.unet_2.up_attn_blocks[1].cross_attn_out, sae4, "up1_cross"),
#     ],
#     diff=True
# )

## Summary

The new `SAEManagerEdit` provides:

1. **Same interface** as old `SAEManager` - just change the import
2. **Direct SAE tracing** - access `sae_block.encoder_out.value` in trace context
3. **Clean interventions** - modify SAE features directly without hooks
4. **Cross-platform** - works on CUDA and MPS

Migration is simple:
```python
# Before
from t2Interp.sae import SAEManager

# After  
from t2Interp.sae_edit import SAEManagerEdit as SAEManager
```